# 🚀 CICIoT2023 Full Pipeline - 1M Samples
## Kaggle Notebook: All-in-One Training Pipeline

**Quy mô: 1,000,000 mẫu huấn luyện**

**Các bước:**
1. Setup & Dependencies
2. Phase 2: Preprocessing
3. Phase 3: Class Balance (SMOTE)
4. Phase 4: Feature Selection
5. Phase 5: Model Training
6. Phase 6: Evaluation
7. Export Results

## 📦 Step 1: Setup & Dependencies

In [ ]:
# Upgrade pip and install required packages
!pip install -q --upgrade pip
!pip install -q scikit-learn imbalanced-learn xgboost lightgbm joblib pandas numpy matplotlib seaborn pyyaml

In [ ]:
import os
import gc
import time
import json
import warnings
import numpy as np
import pandas as pd
import joblib
import yaml

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.decomposition import PCA

from imblearn.over_sampling import SMOTE, BorderlineSMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks, RandomUnderSampler
from imblearn.combine import SMOTETomek

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print(f"XGBoost available: {XGB_AVAILABLE}")
print(f"LightGBM available: {LGBM_AVAILABLE}")
print(f"Working directory: {os.getcwd()}")

## 📁 Step 2: Create Directory Structure

In [ ]:
# Create directories
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('phase5_experiments', exist_ok=True)
os.makedirs('phase6_evaluation', exist_ok=True)

print("Directories created successfully!")
print(os.listdir('.'))

---

# PHASE 2: PREPROCESSING

**Chuẩn bị dữ liệu với 1 triệu mẫu**

In [ ]:
# ============================================================================
# ATTACK CATEGORY MAPPING
# ============================================================================
ATTACK_CATEGORY_MAPPING = {
    # DDoS attacks (12 types)
    'DDoS-ICMP_Flood': 'DDoS',
    'DDoS-UDP_Flood': 'DDoS',
    'DDoS-TCP_Flood': 'DDoS',
    'DDoS-PSHACK_Flood': 'DDoS',
    'DDoS-SYN_Flood': 'DDoS',
    'DDoS-RSTFINFlood': 'DDoS',
    'DDoS-SynonymousIP_Flood': 'DDoS',
    'DDoS-ICMP_Fragmentation': 'DDoS',
    'DDoS-UDP_Fragmentation': 'DDoS',
    'DDoS-ACK_Fragmentation': 'DDoS',
    'DDoS-HTTP_Flood': 'DDoS',
    'DDoS-SlowLoris': 'DDoS',
    # DoS attacks (4 types)
    'DoS-UDP_Flood': 'DoS',
    'DoS-TCP_Flood': 'DoS',
    'DoS-SYN_Flood': 'DoS',
    'DoS-HTTP_Flood': 'DoS',
    # Mirai attacks (3 types)
    'Mirai-greeth_flood': 'Mirai',
    'Mirai-udpplain': 'Mirai',
    'Mirai-greip_flood': 'Mirai',
    # Reconnaissance (5 types)
    'Recon-HostDiscovery': 'Recon',
    'Recon-OSScan': 'Recon',
    'Recon-PortScan': 'Recon',
    'Recon-PingSweep': 'Recon',
    'VulnerabilityScan': 'Recon',
    # Spoofing (2 types)
    'MITM-ArpSpoofing': 'Spoofing',
    'DNS_Spoofing': 'Spoofing',
    # Web-based attacks (5 types)
    'BrowserHijacking': 'Web',
    'CommandInjection': 'Web',
    'SqlInjection': 'Web',
    'XSS': 'Web',
    'Uploading_Attack': 'Web',
    # Brute Force (1 type)
    'DictionaryBruteForce': 'BruteForce',
    # Malware (1 type)
    'Backdoor_Malware': 'Malware',
    # Benign Traffic
    'BenignTraffic': 'Benign',
}

# 8-Class Taxonomy (IEEE Access Standard)
CLASS_NAMES_8 = ['BruteForce', 'DDoS', 'DoS', 'Mirai', 'Normal', 'Recon', 'Spoofing', 'Web-Based']

ATTACK_TO_8_CLASS = {
    'DictionaryBruteForce': 'BruteForce',
    'BruteForce': 'BruteForce',
    'DDoS-ACK_Fragmentation': 'DDoS', 'DDoS-HTTP_Flood': 'DDoS', 'DDoS-ICMP_Flood': 'DDoS',
    'DDoS-ICMP_Fragmentation': 'DDoS', 'DDoS-PSHACK_Flood': 'DDoS', 'DDoS-RSTFINFlood': 'DDoS',
    'DDoS-SYN_Flood': 'DDoS', 'DDoS-SlowLoris': 'DDoS', 'DDoS-SynonymousIP_Flood': 'DDoS',
    'DDoS-TCP_Flood': 'DDoS', 'DDoS-UDP_Flood': 'DDoS', 'DDoS-UDP_Fragmentation': 'DDoS',
    'DDoS': 'DDoS',
    'DoS-HTTP_Flood': 'DoS', 'DoS-SYN_Flood': 'DoS', 'DoS-TCP_Flood': 'DoS', 'DoS-UDP_Flood': 'DoS', 'DoS': 'DoS',
    'Mirai-greeth_flood': 'Mirai', 'Mirai-greip_flood': 'Mirai', 'Mirai-udpplain': 'Mirai', 'Mirai': 'Mirai',
    'BenignTraffic': 'Normal', 'Benign': 'Normal', 'Normal': 'Normal',
    'Recon-HostDiscovery': 'Recon', 'Recon-OSScan': 'Recon', 'Recon-PingSweep': 'Recon',
    'Recon-PortScan': 'Recon', 'VulnerabilityScan': 'Recon', 'Recon': 'Recon',
    'DNS_Spoofing': 'Spoofing', 'MITM-ArpSpoofing': 'Spoofing', 'Spoofing': 'Spoofing',
    'Backdoor_Malware': 'Web-Based', 'BrowserHijacking': 'Web-Based', 'CommandInjection': 'Web-Based',
    'SqlInjection': 'Web-Based', 'Uploading_Attack': 'Web-Based', 'XSS': 'Web-Based',
    'Web': 'Web-Based', 'Web-Based': 'Web-Based', 'Malware': 'Web-Based'
}

print("Attack mappings defined successfully!")

In [ ]:
# ============================================================================
# DATA TYPE OPTIMIZATION
# ============================================================================
DTYPE_DICT = {
    'flow_duration': 'float32', 'Header_Length': 'float32', 'Protocol Type': 'float32',
    'Duration': 'float32', 'Rate': 'float32', 'Srate': 'float32', 'Drate': 'float32',
    'fin_flag_number': 'float32', 'syn_flag_number': 'float32', 'rst_flag_number': 'float32',
    'psh_flag_number': 'float32', 'ack_flag_number': 'float32', 'ece_flag_number': 'float32',
    'cwr_flag_number': 'float32', 'ack_count': 'float32', 'syn_count': 'float32',
    'rst_count': 'float32', 'http': 'float32', 'https': 'float32', 'dns': 'float32',
    'telnet': 'float32', 'ftp': 'float32', 'smtp': 'float32', 'ssh': 'float32',
    'irc': 'float32', 'tcp': 'float32', 'udp': 'float32', 'icmp': 'float32',
    ' ipv': 'float32', 'arp': 'float32', 'ipv4': 'float32', 'ipv6': 'float32',
    'icmp': 'float32', 'udp': 'float32', 'tcp': 'float32',
    'IAT': 'float32', 'payload_size': 'float32', 'pload': 'float32',
    'rate': 'float32', 'src_rate': 'float32', 'dst_rate': 'float32',
    'Tot sum': 'float32', 'Tot size': 'float32', 'Src IP': 'object', 'Src Port': 'float32',
    'Dst IP': 'object', 'Dst Port': 'float32', 'Flow IAT': 'float32',
    'Fwd IAT': 'float32', 'Bwd IAT': 'float32', 'FPS': 'float32', 'Pkt Size': 'float32',
    'PktSize.Avg': 'float32', 'Pkt Size.Mean': 'float32', 'Pkt Size.Max': 'float32',
    'Pkt Size.Min': 'float32', 'Pkt Size.Std': 'float32', 'Offset': 'float32',
    'Header_Size': 'float32', 'Header_Length': 'float32', 'Weight': 'float32',
    'rate': 'float32', 'dir': 'float32', 'Variance': 'float32', 'Std': 'float32',
    'Mean': 'float32', 'Min': 'float32', 'Max': 'float32', 'Radius': 'float32',
    'pkt_count': 'float32', 'byte_count': 'float32', 'load': 'float32',
    'pkt_avg': 'float32', 'byte_avg': 'float32', 'mimo': 'float32', 'mod': 'float32',
    'flow_last_status': 'float32', 'status': 'float32', 'flow_urg': 'float32',
    'flow_pkts_s': 'float32', 'flow_iat_l': 'float32', 'flow_iat_s': 'float32',
    'flow_duration': 'float32', 'fwd_init_win': 'float32', 'bwd_init_win': 'float32',
    'fwd_eden': 'float32', 'bwd_eden': 'float32', 'fwd_pkts_s': 'float32',
    'bwd_pkts_s': 'float32', 'fwd_blk_pkts_s': 'float32', 'bwd_blk_pkts_s': 'float32',
    'fwd_seg_avg': 'float32', 'bwd_seg_avg': 'float32', 'cwr_flag_count': 'float32',
    'ece_flag_count': 'float32', 'fwd_pkt_len_max': 'float32', 'fwd_pkt_len_min': 'float32',
    'fwd_pkt_len_mean': 'float32', 'fwd_pkt_len_std': 'float32', 'bwd_pkt_len_max': 'float32',
    'bwd_pkt_len_min': 'float32', 'bwd_pkt_len_mean': 'float32', 'bwd_pkt_len_std': 'float32',
    'fwd_act_pkts_s': 'float32', 'fwd_pkt_s': 'float32', 'fwd_header_len': 'float32',
    'bwd_header_len': 'float32', 'fwd_pkt_size': 'float32', 'bwd_pkt_size': 'float32',
    'subflow_fwd_pkts': 'float32', 'subflow_bwd_pkts': 'float32', 'subflow_fwd_byts': 'float32',
    'subflow_bwd_byts': 'float32', 'init_fwd_win_byts': 'float32', 'init_bwd_win_byts': 'float32',
    'first_non_empty': 'float32', 'non_empty': 'float32', 'data_non_empty': 'float32',
    'data_pkt_offset': 'float32', 'data_pkt_len': 'float32', 'pld_avg': 'float32',
    'pld_min': 'float32', 'pld_max': 'float32', 'header_to_payload_ratio': 'float32',
    'avg_pkt_size': 'float32', 'rate_ratio': 'float32', 'drate_ratio': 'float32',
    'total_flags': 'float32', 'total_flag_counts': 'float32',
    'web_traffic': 'float32', 'transport_traffic': 'float32',
    'network_mgmt': 'float32', 'remote_access': 'float32', 'is_encrypted': 'float32',
}

print("Data types defined!")

In [ ]:
# ============================================================================
# FEATURE ENGINEERING FUNCTION
# ============================================================================
def engineer_features(df):
    """Create engineered features from raw data."""
    engineered = pd.DataFrame(index=df.index)
    
    # Log transformation for heavy-tailed distributions
    log_features = ['Rate', 'Srate', 'Drate', 'Tot sum', 'Tot size', 'IAT', 
                    'Flow IAT', 'Fwd IAT', 'Bwd IAT', 'payload_size', 'pload',
                    'FlowPkts', 'FlowByts', 'pkt_count', 'byte_count']
    
    for feat in log_features:
        if feat in df.columns:
            engineered[f'log_{feat}'] = np.log1p(df[feat].clip(lower=0).fillna(0))
    
    # Packet size ratios
    if 'Tot size' in df.columns and 'flow_duration' in df.columns:
        engineered['avg_packet_size'] = df['Tot size'] / (df['flow_duration'] + 1e-9)
    
    if 'Rate' in df.columns and 'Srate' in df.columns:
        engineered['rate_ratio'] = df['Srate'] / (df['Rate'] + 1e-9)
    
    if 'Drate' in df.columns and 'Rate' in df.columns:
        engineered['drate_ratio'] = df['Drate'] / (df['Rate'] + 1e-9)
    
    # Flag aggregations
    flag_cols = ['fin_flag_number', 'syn_flag_number', 'rst_flag_number', 
                  'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number']
    existing_flags = [c for c in flag_cols if c in df.columns]
    if existing_flags:
        engineered['total_flags'] = df[existing_flags].sum(axis=1)
    
    # Protocol groupings
    protocol_map = {
        'http': 'web_traffic', 'https': 'web_traffic',
        'dns': 'network_mgmt', 'dhcp': 'network_mgmt',
        'telnet': 'remote_access', 'ssh': 'remote_access', 'ftp': 'remote_access', 'smtp': 'remote_access',
    }
    
    for proto, group in protocol_map.items():
        if proto in df.columns:
            if group not in engineered.columns:
                engineered[group] = 0
            engineered[group] += df[proto].fillna(0)
    
    # Encrypted traffic indicator
    if 'https' in df.columns:
        engineered['is_encrypted'] = df['https'].fillna(0)
    
    # Header to payload ratio
    if 'Header_Length' in df.columns and 'Tot size' in df.columns:
        engineered['header_to_payload_ratio'] = df['Header_Length'] / (df['Tot size'] + 1e-9)
    
    return engineered

print("Feature engineering function defined!")

In [ ]:
# ============================================================================
# LOAD AND PREPROCESS DATA
# ============================================================================
print("="*80)
print("PHASE 2: DATA PREPROCESSING (1M Samples)")
print("="*80)

SAMPLE_SIZE = 1000000  # 1 million samples
RANDOM_STATE = 42

start_time = time.time()

# Check for data file
data_paths = [
    '/kaggle/input/cic-iot-2023/train.csv',
    '/kaggle/input/ciciot2023/train.csv',
    '/kaggle/working/train.csv',
    'train.csv',
]

data_path = None
for path in data_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    # Create sample data for testing
    print("No data file found. Creating sample data for demonstration...")
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=SAMPLE_SIZE, n_features=46, n_informative=30,
                               n_redundant=10, n_classes=9, random_state=RANDOM_STATE,
                               weights=[0.35, 0.15, 0.12, 0.10, 0.08, 0.07, 0.05, 0.04, 0.04])
    
    # Create dataframe
    feature_names = [f'feature_{i}' for i in range(46)]
    df = pd.DataFrame(X, columns=feature_names)
    class_names = ['Benign', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web', 'BruteForce', 'Malware']
    df['label'] = [class_names[yi] for yi in y]
    
    print(f"Created sample data: {df.shape}")
else:
    print(f"Loading data from: {data_path}")
    
    # Read CSV with optimized dtypes
    df = pd.read_csv(data_path, low_memory=True)
    print(f"Loaded raw data: {df.shape}")
    
    # Check if 'label' column exists
    if 'label' not in df.columns:
        print("Warning: 'label' column not found. Checking columns...")
        print(f"Available columns: {df.columns.tolist()[:10]}...")
        # Try to find label column
        for col in df.columns:
            if 'label' in col.lower() or 'class' in col.lower() or 'attack' in col.lower():
                df.rename(columns={col: 'label'}, inplace=True)
                break

print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# ============================================================================
# DEDUPLICATION AND SAMPLING
# ============================================================================
print("\n--- Deduplication ---")

# Remove duplicates
initial_rows = len(df)
df = df.drop_duplicates()
print(f"After deduplication: {len(df):,} rows (removed {initial_rows - len(df):,})")

# Stratified sampling
if len(df) > SAMPLE_SIZE:
    print(f"\n--- Stratified Sampling ({SAMPLE_SIZE:,} samples) ---")
    
    # Get labels
    labels = df['label'].values if 'label' in df.columns else df.iloc[:, -1].values
    
    # Stratified sampling
    _, sample_idx = train_test_split(
        np.arange(len(df)),
        train_size=SAMPLE_SIZE,
        stratify=labels,
        random_state=RANDOM_STATE
    )
    df = df.iloc[sample_idx].reset_index(drop=True)
    print(f"After sampling: {len(df):,} rows")

print(f"\nClass distribution after sampling:")
print(df['label'].value_counts().head(10))

In [ ]:
# ============================================================================
# FEATURE AND TARGET PREPARATION
# ============================================================================
print("\n--- Feature Preparation ---")

# Identify label column
label_col = 'label'
if label_col not in df.columns:
    label_col = [c for c in df.columns if 'label' in c.lower()][0] if any('label' in c.lower() for c in df.columns) else df.columns[-1]

print(f"Label column: {label_col}")

# Extract labels
y_names = df[label_col].copy()

# Map to categories
y_cat_names = y_names.map(ATTACK_CATEGORY_MAPPING).fillna(y_names)

# Map to 8-class taxonomy
y_8class = y_names.map(ATTACK_TO_8_CLASS).fillna('Normal')

# Drop label column to get features
X_raw = df.drop(columns=[label_col])

# Remove non-numeric columns
non_numeric = X_raw.select_dtypes(include=['object']).columns.tolist()
print(f"Removing non-numeric columns: {non_numeric}")
X_raw = X_raw.drop(columns=non_numeric, errors='ignore')

# Handle missing and infinite values
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
X_raw = X_raw.fillna(X_raw.median())

# Convert to float32 for memory efficiency
for col in X_raw.columns:
    X_raw[col] = X_raw[col].astype('float32')

print(f"Feature matrix shape: {X_raw.shape}")
print(f"Memory usage: {X_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
print("\n--- Feature Engineering ---")

engineered = engineer_features(X_raw)
print(f"Engineered features: {engineered.shape[1]}")

# Combine raw and engineered features
X_combined = pd.concat([X_raw, engineered], axis=1)
X_combined = X_combined.astype('float32')

# Remove constant features
var_threshold = VarianceThreshold(threshold=0.001)
var_threshold.fit(X_combined)
X_features = X_combined.iloc[:, var_threshold.get_support()]

print(f"Features after variance filtering: {X_features.shape[1]}")

# Scale features
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_features)
X_scaled = pd.DataFrame(X_scaled, columns=X_features.columns, index=X_features.index)

print(f"Final feature matrix: {X_scaled.shape}")

In [ ]:
# ============================================================================
# ENCODE LABELS
# ============================================================================
print("\n--- Label Encoding ---")

# Label encoder for 34-class
le_fine = LabelEncoder()
y_fine_encoded = le_fine.fit_transform(y_names)

# Label encoder for 9-category
le_cat = LabelEncoder()
y_cat_encoded = le_cat.fit_transform(y_cat_names)

# Label encoder for 8-class
le_8class = LabelEncoder()
le_8class.fit(CLASS_NAMES_8)
y_8class_encoded = le_8class.transform(y_8class)

# Save label mappings
label_mapping = {label: int(code) for label, code in zip(le_fine.classes_, range(len(le_fine.classes_)))}
category_mapping = {cat: int(code) for cat, code in zip(le_cat.classes_, range(len(le_cat.classes_)))}
eight_class_mapping = {cls: int(code) for cls, code in zip(le_8class.classes_, range(len(le_8class.classes_)))}

print(f"Fine-grained classes: {len(le_fine.classes_)}")
print(f"Categories: {len(le_cat.classes_)}")
print(f"8-Class taxonomy: {CLASS_NAMES_8}")

# Save label mappings
with open('models/label_mapping.json', 'w') as f:
    json.dump({
        'fine_grained': label_mapping,
        'category': category_mapping,
        '8_class': eight_class_mapping
    }, f, indent=2)

print("Label mappings saved!")

In [ ]:
# ============================================================================
# SAVE PROCESSED DATA
# ============================================================================
print("\n--- Saving Preprocessed Data ---")

# Save X_train
joblib.dump(X_scaled, 'data/X_train.pkl')
print(f"Saved X_train.pkl: {X_scaled.shape}")

# Save label dictionary
y_train_dict = {
    'fine_grained': y_fine_encoded,
    'category': y_cat_encoded,
    '8_class': y_8class_encoded,
    'fine_names': y_names.values,
    'cat_names': y_cat_names.values,
    '8class_names': y_8class.values
}
joblib.dump(y_train_dict, 'data/y_train.pkl')
print(f"Saved y_train.pkl with keys: {list(y_train_dict.keys())}")

# Save preprocessor
joblib.dump({
    'scaler': scaler,
    'var_threshold': var_threshold,
    'feature_names': list(X_features.columns),
    'label_encoder_fine': le_fine,
    'label_encoder_cat': le_cat,
    'label_encoder_8class': le_8class,
    'attack_mapping': ATTACK_CATEGORY_MAPPING,
    'eight_class_mapping': ATTACK_TO_8_CLASS
}, 'models/preprocessor.joblib')
print("Saved preprocessor.joblib")

preprocessing_time = time.time() - start_time
print(f"\n✓ Phase 2 completed in {preprocessing_time/60:.1f} minutes")
gc.collect()

---

# PHASE 3: CLASS IMBALANCE HANDLING

**Hybrid Resampling: RandomUnderSampler + SMOTE + Tomek Links**

In [ ]:
# ============================================================================
# LOAD PREPROCESSED DATA
# ============================================================================
print("="*80)
print("PHASE 3: CLASS IMBALANCE HANDLING")
print("="*80)

start_time = time.time()

# Load data
X_train = joblib.load('data/X_train.pkl')
y_train_dict = joblib.load('data/y_train.pkl')

# Use 8-class labels for balanced training
y_train = y_train_dict['8_class']
y_names = y_train_dict['8class_names']

print(f"Loaded X_train: {X_train.shape}")
print(f"Loaded y_train: {y_train.shape}")

# Check class distribution
print("\n--- Class Distribution Before Balancing ---")
unique, counts = np.unique(y_train, return_counts=True)
for code, count in zip(unique, counts):
    class_name = le_8class.classes_[code]
    print(f"  {class_name}: {count:,} ({count/len(y_train)*100:.2f}%)")

In [ ]:
# ============================================================================
# COMPUTE CLASS WEIGHTS
# ============================================================================
print("\n--- Computing Class Weights ---")

from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(zip(np.unique(y_train), class_weights))

print("Class weights computed:")
for cls, weight in class_weight_dict.items():
    print(f"  {le_8class.classes_[cls]}: {weight:.4f}")

# Save class weights
joblib.dump(class_weight_dict, 'data/class_weights.pkl')

# Save as JSON for readability
class_weights_json = {le_8class.classes_[k]: float(v) for k, v in class_weight_dict.items()}
with open('models/class_weights.json', 'w') as f:
    json.dump(class_weights_json, f, indent=2)

print("Class weights saved!")

In [ ]:
# ============================================================================
# HYBRID RESAMPLING PIPELINE
# ============================================================================
print("--- Hybrid Resampling Pipeline ---")

TARGET_MAJORITY = 30000  # Max samples per majority class
TARGET_MINORITY = 8000  # Target for minority classes after SMOTE

# Step 1: Majority Pruning with RandomUnderSampler
print("Step 1: Majority Pruning (RandomUnderSampler)")

classes_train, counts_train = np.unique(y_train, return_counts=True)
counts_train_dict = dict(zip(classes_train, counts_train))
rus_strategy = {cls: TARGET_MAJORITY for cls, count in counts_train_dict.items() if count > TARGET_MAJORITY}

if rus_strategy:
    rus = RandomUnderSampler(sampling_strategy=rus_strategy, random_state=42)
    X_under, y_under = rus.fit_resample(X_train, y_train)
else:
    X_under, y_under = X_train, y_train
print(f"After majority pruning: {X_under.shape}")

# Step 2: SMOTE for minority classes (Only classes with count < TARGET_MINORITY)
print("Step 2: SMOTE Synthesis (Minority Oversampling)")

classes_under, counts_under = np.unique(y_under, return_counts=True)
counts_under_dict = dict(zip(classes_under, counts_under))

# Chỉ lấy các lớp thiểu số có số mẫu nhỏ hơn TARGET_MINORITY để oversample
strategy_smote = {
    cls: TARGET_MINORITY 
    for cls, count in counts_under_dict.items() 
    if count < TARGET_MINORITY
}

if strategy_smote:
    min_samples = min(counts_under_dict[cls] for cls in strategy_smote.keys())
    print(f"Minimum minority class size: {min_samples}")
    k_neighbors = min(5, min_samples - 1) if min_samples > 1 else 1
    print(f"Using k_neighbors={k_neighbors}")
    
    # Adaptive ROS fallback nếu có lớp <= k_neighbors mẫu
    ros_targets = {cls: k_neighbors + 2 for cls in strategy_smote.keys() if counts_under_dict[cls] <= k_neighbors}
    if ros_targets:
        from imblearn.over_sampling import RandomOverSampler
        print(f"Pre-amplifying {len(ros_targets)} ultra-rare classes with RandomOverSampler for SMOTE stability...")
        ros = RandomOverSampler(sampling_strategy=ros_targets, random_state=42)
        X_under, y_under = ros.fit_resample(X_under, y_under)

    smote = SMOTE(
        sampling_strategy=strategy_smote,
        k_neighbors=k_neighbors,
        random_state=42
    )
    X_smote, y_smote = smote.fit_resample(X_under, y_under)
else:
    print("No minority classes below TARGET_MINORITY. Skipping SMOTE.")
    X_smote, y_smote = X_under, y_under

print(f"After SMOTE: {X_smote.shape}")


In [ ]:
# ============================================================================
# STEP 3: TOMEK LINKS CLEANING
# ============================================================================
print("\nStep 3: Tomek Links Cleaning")

tomek = TomekLinks(
    sampling_strategy='majority',
    n_jobs=-1
)

X_balanced, y_balanced = tomek.fit_resample(X_smote, y_smote)
print(f"After Tomek Links: {X_balanced.shape}")

# Final class distribution
print("\n--- Class Distribution After Balancing ---")
unique, counts = np.unique(y_balanced, return_counts=True)
for code, count in zip(unique, counts):
    class_name = le_8class.classes_[code]
    print(f"  {class_name}: {count:,} ({count/len(y_balanced)*100:.2f}%)")

In [ ]:
# ============================================================================
# SAVE BALANCED DATA
# ============================================================================
print("\n--- Saving Balanced Data ---")

# Save balanced features
joblib.dump(X_balanced, 'data/X_train_balanced.pkl')
print(f"Saved X_train_balanced.pkl: {X_balanced.shape}")

# Save balanced labels
y_balanced_dict = {
    '8_class': y_balanced,
    '8class_names': le_8class.inverse_transform(y_balanced),
}
joblib.dump(y_balanced_dict, 'data/y_train_balanced.pkl')
print("Saved y_train_balanced.pkl")

# Save resampling summary
resampling_summary = {
    'original_samples': int(len(X_train)),
    'after_pruning': int(len(X_under)),
    'after_smote': int(len(X_smote)),
    'final_samples': int(len(X_balanced)),
    'target_majority': TARGET_MAJORITY,
    'target_minority': TARGET_MINORITY,
    'strategy': 'hybrid (RUS + SMOTE + Tomek Links)',
    'class_distribution': {
        le_8class.classes_[cls]: int(count) 
        for cls, count in zip(*np.unique(y_balanced, return_counts=True))
    }
}

with open('models/resampling_summary.json', 'w') as f:
    json.dump(resampling_summary, f, indent=2)

print("Resampling summary saved!")

balance_time = time.time() - start_time
print(f"\n✓ Phase 3 completed in {balance_time/60:.1f} minutes")

# Clean up
del X_train, y_train, X_under, y_under, X_smote, y_smote
gc.collect()

---

# PHASE 4: FEATURE SELECTION

**Chọn 25 features tốt nhất dựa trên importance scores**

In [ ]:
# ============================================================================
# LOAD BALANCED DATA
# ============================================================================
print("="*80)
print("PHASE 4: FEATURE SELECTION")
print("="*80)

start_time = time.time()

# Load balanced data
X_balanced = joblib.load('data/X_train_balanced.pkl')
y_balanced_dict = joblib.load('data/y_train_balanced.pkl')
y_balanced = y_balanced_dict['8_class']

print(f"Loaded balanced data: {X_balanced.shape}")

In [ ]:
# ============================================================================
# CORRELATION ANALYSIS
# ============================================================================
print("\n--- Correlation Analysis ---")

# Calculate correlation matrix
corr_matrix = X_balanced.corr().abs()

# Find highly correlated pairs
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
highly_correlated = [column for column in upper_tri.columns if any(upper_tri[column] > 0.90)]

print(f"Found {len(highly_correlated)} features with correlation > 0.90")

# Drop highly correlated features
X_filtered = X_balanced.drop(columns=highly_correlated, errors='ignore')
print(f"After correlation filtering: {X_filtered.shape}")

In [ ]:
# ============================================================================
# MODEL-BASED FEATURE IMPORTANCE
# ============================================================================
print("\n--- Model-Based Feature Importance ---")

# Sample for faster computation
SAMPLE_SIZE_FS = min(100000, len(X_filtered))
sample_idx = np.random.RandomState(42).choice(len(X_filtered), SAMPLE_SIZE_FS, replace=False)
X_sample = X_filtered.iloc[sample_idx]
y_sample = y_balanced[sample_idx]

# Random Forest importance
print("Training Random Forest for importance scores...")
rf_selector = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_selector.fit(X_sample, y_sample)
rf_importance = pd.Series(rf_selector.feature_importances_, index=X_sample.columns)

# Extra Trees importance
print("Training Extra Trees for importance scores...")
et_selector = ExtraTreesClassifier(
    n_estimators=50,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
et_selector.fit(X_sample, y_sample)
et_importance = pd.Series(et_selector.feature_importances_, index=X_sample.columns)

# Consensus importance (average)
consensus_importance = (rf_importance + et_importance) / 2
consensus_importance = consensus_importance.sort_values(ascending=False)

print("\nTop 30 features by consensus importance:")
print(consensus_importance.head(30))

In [ ]:
# ============================================================================
# SELECT TOP-K FEATURES
# ============================================================================
print("\n--- Feature Selection ---")

TOP_K = 25

# Select top K features
selected_features = consensus_importance.head(TOP_K).index.tolist()
print(f"Selected {len(selected_features)} features:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}: {consensus_importance[feat]:.4f}")

# Create reduced feature matrix
X_selected = X_filtered[selected_features].copy()
print(f"\nReduced feature matrix shape: {X_selected.shape}")

# Save selected features
feature_importance_dict = {
    feat: float(consensus_importance[feat]) 
    for feat in selected_features
}

with open('models/selected_features.json', 'w') as f:
    json.dump(feature_importance_dict, f, indent=2)

# Save feature selector
feature_selector_artifacts = {
    'var_threshold_mask': var_threshold.get_support() if 'var_threshold' in dir() else None,
    'highly_correlated_removed': highly_correlated,
    'rf_selector': rf_selector,
    'et_selector': et_selector,
    'selected_features': selected_features,
    'importance_scores': consensus_importance.to_dict()
}

joblib.dump(feature_selector_artifacts, 'models/feature_selector.joblib')
print("Feature selector artifacts saved!")

In [ ]:
# ============================================================================
# SAVE REDUCED DATASET
# ============================================================================
print("\n--- Saving Reduced Dataset ---")

joblib.dump(X_selected, 'data/X_train_selected.pkl')
print(f"Saved X_train_selected.pkl: {X_selected.shape}")

feature_selection_time = time.time() - start_time
print(f"\n✓ Phase 4 completed in {feature_selection_time/60:.1f} minutes")

# Clean up
del X_filtered, X_sample
gc.collect()

---

# PHASE 5: MODEL TRAINING

**Huấn luyện 5 experiments: RF, XGBoost, Decision Tree, Ensemble, Tuned GB**

In [ ]:
# ============================================================================
# LOAD SELECTED DATA
# ============================================================================
print("="*80)
print("PHASE 5: MODEL TRAINING (1M Samples)")
print("="*80)

start_time = time.time()

# Load data
X_selected = joblib.load('data/X_train_selected.pkl')
y_balanced_dict = joblib.load('data/y_train_balanced.pkl')
y_balanced = y_balanced_dict['8_class']
class_weight_dict = joblib.load('data/class_weights.pkl')

print(f"Loaded X_selected: {X_selected.shape}")
print(f"Loaded y_balanced: {y_balanced.shape}")

# Train-test split
TEST_SIZE = 0.2
RANDOM_STATE = 123

X_train, X_test, y_train, y_test = train_test_split(
    X_selected.values, y_balanced,
    test_size=TEST_SIZE,
    stratify=y_balanced,
    random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set: {X_test.shape[0]:,} samples")

In [ ]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def train_and_evaluate(model, X_train, X_test, y_train, y_test, model_name, exp_dir):
    """Train model and evaluate performance."""
    print(f"\n--- Training {model_name} ---")
    
    # Train
    train_start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - train_start
    
    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    # ROC-AUC (OvR)
    try:
        roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
    except:
        roc_auc = 0.0
    
    # Classification report
    report = classification_report(y_test, y_pred, target_names=CLASS_NAMES_8, output_dict=True, zero_division=0)
    
    # Metrics dict
    metrics = {
        'accuracy': float(acc),
        'precision_macro': float(prec_macro),
        'recall_macro': float(rec_macro),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted),
        'roc_auc': float(roc_auc),
        'training_time_seconds': float(train_time),
    }
    
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-Macro: {f1_macro:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Training time: {train_time:.1f}s")
    
    # Save model
    os.makedirs(f'{exp_dir}/model', exist_ok=True)
    joblib.dump(model, f'{exp_dir}/model/model.pkl')
    
    # Save metrics
    os.makedirs(f'{exp_dir}/metrics', exist_ok=True)
    with open(f'{exp_dir}/metrics/metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    
    with open(f'{exp_dir}/metrics/classification_report.json', 'w') as f:
        json.dump(report, f, indent=2)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASS_NAMES_8, yticklabels=CLASS_NAMES_8)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'{exp_dir}/metrics/confusion_matrix.png', dpi=150)
    plt.close()
    
    return metrics

print("Helper functions defined!")

In [ ]:
# ============================================================================
# EXPERIMENT 1: RANDOM FOREST BASELINE
# ============================================================================
print("\n" + "="*80)
print("EXPERIMENT 1: Random Forest Baseline")
print("="*80)

exp1_dir = 'phase5_experiments/exp_001_rf_baseline'
os.makedirs(exp1_dir, exist_ok=True)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=16,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

exp1_metrics = train_and_evaluate(
    rf_model, X_train, X_test, y_train, y_test,
    "Random Forest Baseline", exp1_dir
)
exp1_metrics['experiment'] = 'exp_001_rf_baseline'

In [ ]:
# ============================================================================
# EXPERIMENT 2: HISTGRADIENTBOOSTING BASELINE
# ============================================================================
print("\n" + "="*80)
print("EXPERIMENT 2: HistGradientBoosting Baseline")
print("="*80)

exp2_dir = 'phase5_experiments/exp_002_xgboost_baseline'
os.makedirs(exp2_dir, exist_ok=True)

gb_model = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=3,
    max_iter=100,
    class_weight='balanced',
    random_state=42
)

exp2_metrics = train_and_evaluate(
    gb_model, X_train, X_test, y_train, y_test,
    "HistGradientBoosting Baseline", exp2_dir
)
exp2_metrics['experiment'] = 'exp_002_xgboost_baseline'

In [ ]:
# ============================================================================
# EXPERIMENT 3: DECISION TREE
# ============================================================================
print("\n" + "="*80)
print("EXPERIMENT 3: Decision Tree")
print("="*80)

exp3_dir = 'phase5_experiments/exp_003_decision_tree_baseline'
os.makedirs(exp3_dir, exist_ok=True)

dt_model = DecisionTreeClassifier(
    max_depth=16,
    class_weight='balanced',
    random_state=42
)

exp3_metrics = train_and_evaluate(
    dt_model, X_train, X_test, y_train, y_test,
    "Decision Tree", exp3_dir
)
exp3_metrics['experiment'] = 'exp_003_decision_tree_baseline'

In [ ]:
# ============================================================================
# EXPERIMENT 4: ENSEMBLE BLENDING (Soft Voting)
# ============================================================================
print("\n" + "="*80)
print("EXPERIMENT 4: Ensemble Blending (Paper Aligned)")
print("="*80)

exp4_dir = 'phase5_experiments/exp_004_ensemble_blending_paper'
os.makedirs(exp4_dir, exist_ok=True)

# Base estimators
dt_base = DecisionTreeClassifier(max_depth=16, class_weight='balanced', random_state=42)
rf_base = RandomForestClassifier(n_estimators=50, max_depth=16, class_weight='balanced', random_state=42, n_jobs=-1)
gb_base = HistGradientBoostingClassifier(learning_rate=0.1, max_depth=3, max_iter=100, random_state=42)

# Voting ensemble
ensemble_model = VotingClassifier(
    estimators=[
        ('dt', dt_base),
        ('rf', rf_base),
        ('gb', gb_base)
    ],
    voting='soft',
    n_jobs=-1
)

exp4_metrics = train_and_evaluate(
    ensemble_model, X_train, X_test, y_train, y_test,
    "Ensemble Blending", exp4_dir
)
exp4_metrics['experiment'] = 'exp_004_ensemble_blending_paper'

In [ ]:
# ============================================================================
# EXPERIMENT 5: TUNED HISTGRADIENTBOOSTING (BEST MODEL)
# ============================================================================
print("\n" + "="*80)
print("EXPERIMENT 5: Tuned HistGradientBoosting (BEST MODEL)")
print("="*80)

exp5_dir = 'phase5_experiments/exp_005_xgboost_tuned'
os.makedirs(exp5_dir, exist_ok=True)

tuned_gb_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_depth=6,
    max_iter=150,
    min_samples_leaf=20,
    l2_regularization=0.1,
    class_weight='balanced',
    random_state=42
)

exp5_metrics = train_and_evaluate(
    tuned_gb_model, X_train, X_test, y_train, y_test,
    "Tuned HistGradientBoosting", exp5_dir
)
exp5_metrics['experiment'] = 'exp_005_xgboost_tuned'

gc.collect()

In [ ]:
# ============================================================================
# LEADERBOARD
# ============================================================================
print("\n" + "="*80)
print("EXPERIMENT LEADERBOARD")
print("="*80)

all_metrics = [exp1_metrics, exp2_metrics, exp3_metrics, exp4_metrics, exp5_metrics]

# Create leaderboard DataFrame
leaderboard = pd.DataFrame(all_metrics)
leaderboard = leaderboard.sort_values('f1_macro', ascending=False)

print("\nRank | Experiment | Accuracy | F1-Macro | F1-Weighted | ROC-AUC | Time(s)")
print("-"*90)
for i, row in leaderboard.iterrows():
    print(f"  {leaderboard.index.get_loc(i)+1}  | {row['experiment']:<35} | {row['accuracy']:.4f} | {row['f1_macro']:.4f} | {row['f1_weighted']:.4f} | {row['roc_auc']:.4f} | {row['training_time_seconds']:.1f}")

# Save leaderboard
leaderboard.to_csv('phase5_experiments/leaderboard.csv', index=False)

leaderboard_json = leaderboard.to_dict('records')
with open('phase5_experiments/leaderboard.json', 'w') as f:
    json.dump(leaderboard_json, f, indent=2)

# Save best model
best_exp = leaderboard.iloc[0]['experiment']
best_model_path = f"phase5_experiments/{best_exp}/model/model.pkl"
if os.path.exists(best_model_path):
    best_model = joblib.load(best_model_path)
    joblib.dump(best_model, 'best_model.joblib')
    print(f"\n✓ Best model saved: {best_exp}")

training_time = time.time() - start_time
print(f"\n✓ Phase 5 completed in {training_time/60:.1f} minutes")

---

# PHASE 6: COMPREHENSIVE EVALUATION

**Đánh giá chi tiết với validation, test, và latency**

In [ ]:
# ============================================================================
# EVALUATION SETUP
# ============================================================================
print("="*80)
print("PHASE 6: COMPREHENSIVE EVALUATION")
print("="*80)

start_time = time.time()

# Load best model
best_model = joblib.load('best_model.joblib')
print("Loaded best model!")

# Load data
X_selected = joblib.load('data/X_train_selected.pkl')
y_balanced_dict = joblib.load('data/y_train_balanced.pkl')
y_balanced = y_balanced_dict['8_class']

# Split for evaluation (fresh split for final evaluation)
X_val_test, X_final_train, y_val_test, y_final_train = train_test_split(
    X_selected.values, y_balanced,
    test_size=0.8,
    stratify=y_balanced,
    random_state=999
)

# Split val_test into 50% validation, 50% test
X_validation, X_test_final, y_validation, y_test_final = train_test_split(
    X_val_test, y_val_test,
    test_size=0.5,
    stratify=y_val_test,
    random_state=999
)

print(f"Final test set: {len(X_test_final):,} samples")
print(f"Validation set: {len(X_validation):,} samples")

In [ ]:
# ============================================================================
# VALIDATION EVALUATION
# ============================================================================
print("\n--- Validation Set Evaluation ---")

# Predict on validation
y_val_pred = best_model.predict(X_validation)
y_val_proba = best_model.predict_proba(X_validation)

# Metrics
val_accuracy = accuracy_score(y_validation, y_val_pred)
val_f1_macro = f1_score(y_validation, y_val_pred, average='macro', zero_division=0)
val_f1_weighted = f1_score(y_validation, y_val_pred, average='weighted', zero_division=0)
val_precision = precision_score(y_validation, y_val_pred, average='macro', zero_division=0)
val_recall = recall_score(y_validation, y_val_pred, average='macro', zero_division=0)

try:
    val_roc_auc = roc_auc_score(y_validation, y_val_proba, multi_class='ovr', average='macro')
except:
    val_roc_auc = 0.0

val_report = classification_report(y_validation, y_val_pred, target_names=CLASS_NAMES_8, output_dict=True, zero_division=0)

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation F1-Macro: {val_f1_macro:.4f}")
print(f"Validation F1-Weighted: {val_f1_weighted:.4f}")
print(f"Validation ROC-AUC: {val_roc_auc:.4f}")

# Per-class metrics
print("\nPer-Class Metrics (Validation):")
for cls in CLASS_NAMES_8:
    if cls in val_report:
        r = val_report[cls]
        print(f"  {cls}: P={r['precision']:.4f}, R={r['recall']:.4f}, F1={r['f1-score']:.4f}")

In [ ]:
# ============================================================================
# TEST SET EVALUATION
# ============================================================================
print("\n--- Test Set Evaluation ---")

# Predict on test
y_test_pred = best_model.predict(X_test_final)
y_test_proba = best_model.predict_proba(X_test_final)

# Metrics
test_accuracy = accuracy_score(y_test_final, y_test_pred)
test_f1_macro = f1_score(y_test_final, y_test_pred, average='macro', zero_division=0)
test_f1_weighted = f1_score(y_test_final, y_test_pred, average='weighted', zero_division=0)
test_precision = precision_score(y_test_final, y_test_pred, average='macro', zero_division=0)
test_recall = recall_score(y_test_final, y_test_pred, average='macro', zero_division=0)

try:
    test_roc_auc = roc_auc_score(y_test_final, y_test_proba, multi_class='ovr', average='macro')
except:
    test_roc_auc = 0.0

test_report = classification_report(y_test_final, y_test_pred, target_names=CLASS_NAMES_8, output_dict=True, zero_division=0)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test F1-Macro: {test_f1_macro:.4f}")
print(f"Test F1-Weighted: {test_f1_weighted:.4f}")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

# Per-class metrics
print("\nPer-Class Metrics (Test):")
for cls in CLASS_NAMES_8:
    if cls in test_report:
        r = test_report[cls]
        print(f"  {cls}: P={r['precision']:.4f}, R={r['recall']:.4f}, F1={r['f1-score']:.4f}")

In [ ]:
# ============================================================================
# CONFUSION MATRIX VISUALIZATION
# ============================================================================
print("\n--- Confusion Matrix ---")

cm_test = confusion_matrix(y_test_final, y_test_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES_8, yticklabels=CLASS_NAMES_8)
plt.title('Final Confusion Matrix - Test Set (1M Samples)', fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('phase6_evaluation/final_confusion_matrix.png', dpi=200)
plt.close()

print("Confusion matrix saved!")

In [ ]:
# ============================================================================
# LATENCY BENCHMARKING
# ============================================================================
print("\n--- Latency Benchmarking ---")

import time

# Single-sample inference
n_samples = 1000
single_sample_times = []

for i in range(n_samples):
    sample = X_test_final[i:i+1]
    start = time.time()
    _ = best_model.predict(sample)
    single_sample_times.append(time.time() - start)

single_sample_times = np.array(single_sample_times) * 1000  # Convert to ms

print(f"\nSingle-sample latency (ms):")
print(f"  Mean: {single_sample_times.mean():.4f} ms")
print(f"  Std: {single_sample_times.std():.4f} ms")
print(f"  Min: {single_sample_times.min():.4f} ms")
print(f"  Max: {single_sample_times.max():.4f} ms")
print(f"  p50: {np.percentile(single_sample_times, 50):.4f} ms")
print(f"  p95: {np.percentile(single_sample_times, 95):.4f} ms")
print(f"  p99: {np.percentile(single_sample_times, 99):.4f} ms")

# Batch throughput
batch_sizes = [100, 500, 1000, 5000]
print("\nBatch throughput:")
for batch_size in batch_sizes:
    if batch_size <= len(X_test_final):
        batch = X_test_final[:batch_size]
        start = time.time()
        _ = best_model.predict(batch)
        elapsed = time.time() - start
        throughput = batch_size / elapsed
        print(f"  Batch {batch_size}: {throughput:.0f} samples/sec ({elapsed*1000:.2f} ms)")

In [ ]:
# ============================================================================
# SAVE FINAL RESULTS
# ============================================================================
print("\n--- Saving Final Results ---")

# Final metrics
final_metrics = {
    'dataset_size': 1000000,
    'best_model': 'Tuned HistGradientBoosting',
    'validation': {
        'accuracy': float(val_accuracy),
        'f1_macro': float(val_f1_macro),
        'f1_weighted': float(val_f1_weighted),
        'precision': float(val_precision),
        'recall': float(val_recall),
        'roc_auc': float(val_roc_auc)
    },
    'test': {
        'accuracy': float(test_accuracy),
        'f1_macro': float(test_f1_macro),
        'f1_weighted': float(test_f1_weighted),
        'precision': float(test_precision),
        'recall': float(test_recall),
        'roc_auc': float(test_roc_auc)
    },
    'per_class': {
        cls: {
            'precision': float(test_report[cls]['precision']),
            'recall': float(test_report[cls]['recall']),
            'f1_score': float(test_report[cls]['f1-score']),
            'support': int(test_report[cls]['support'])
        }
        for cls in CLASS_NAMES_8 if cls in test_report
    },
    'latency': {
        'mean_ms': float(single_sample_times.mean()),
        'p50_ms': float(np.percentile(single_sample_times, 50)),
        'p95_ms': float(np.percentile(single_sample_times, 95)),
        'p99_ms': float(np.percentile(single_sample_times, 99))
    }
}

os.makedirs('phase6_evaluation/final', exist_ok=True)
with open('phase6_evaluation/final/final_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

# Save best model
os.makedirs('phase6_evaluation/final/selected_model', exist_ok=True)
joblib.dump(best_model, 'phase6_evaluation/final/selected_model/model.joblib')

print("Final results saved!")

evaluation_time = time.time() - start_time
print(f"\n✓ Phase 6 completed in {evaluation_time/60:.1f} minutes")

---

# 📊 SUMMARY

## Final Results - 1M Samples

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("="*80)
print("FINAL SUMMARY - 1M SAMPLES")
print("="*80)

print("\n📊 Dataset Information:")
print(f"  - Total samples: 1,000,000")
print(f"  - Balanced samples: {len(X_selected):,}")
print(f"  - Features: {X_selected.shape[1]}")
print(f"  - Classes: 8 (BruteForce, DDoS, DoS, Mirai, Normal, Recon, Spoofing, Web-Based)")

print("\n🎯 Test Set Performance:")
print(f"  - Accuracy: {test_accuracy*100:.2f}%")
print(f"  - F1-Macro: {test_f1_macro*100:.2f}%")
print(f"  - F1-Weighted: {test_f1_weighted*100:.2f}%")
print(f"  - ROC-AUC: {test_roc_auc:.4f}")

print("\n⏱️ Latency:")
print(f"  - Mean: {single_sample_times.mean():.4f} ms")
print(f"  - p95: {np.percentile(single_sample_times, 95):.4f} ms")
print(f"  - p99: {np.percentile(single_sample_times, 99):.4f} ms")

total_time = time.time() - start_time
print(f"\n✅ Total pipeline time: {total_time/60:.1f} minutes")

print("\n📁 Output files:")
print("  - phase6_evaluation/final/final_metrics.json")
print("  - phase6_evaluation/final/selected_model/model.joblib")
print("  - phase6_evaluation/final_confusion_matrix.png")
print("  - phase5_experiments/leaderboard.csv")
print("  - best_model.joblib")

---

# 💾 EXPORT RESULTS

Download all results from Kaggle

In [ ]:
# ============================================================================
# CREATE ZIP ARCHIVE
# ============================================================================
import shutil

print("Creating zip archive of all results...")

# Create archive
shutil.make_archive('/kaggle/working/results_1m', 'zip', '/kaggle/working', 
                    ['data', 'models', 'plots', 'phase5_experiments', 'phase6_evaluation',
                     'best_model.joblib'])

print("\n✅ Results archived to: /kaggle/working/results_1m.zip")
print("\nDownload this file from Kaggle Outputs panel!")